# Lab 1 · Exercise B — LHCb mass peak and the central limit theorem
**B1.** Simplified dataset modelled on an LHCb analysis: histogram of a B-meson invariant mass (`examples/lhcb-b0-ppbar/data/b0_invariant_mass.root`, histogram `massaB0`).
**B2.** Sum of N uniform variables: where does the Gaussian approximation stop working?

In [ ]:
import ROOT
%jsroot on   # interactive drawing in the notebook (needed on Binder)
import numpy as np

MATRICOLA = 123456   # TODO: write your student ID number here (digits only)
assert MATRICOLA != 0, "Set MATRICOLA first: it seeds YOUR random toy data"

ROOT.RooRandom.randomGenerator().SetSeed(MATRICOLA)
ROOT.RooMsgService.instance().setGlobalKillBelow(ROOT.RooFit.WARNING)

def check(name, ok):
    """Minimal self-check: prints PASS/FAIL and stops the notebook on FAIL."""
    print(("PASS  " if ok else "FAIL  ") + name)
    assert ok, name

## B1 · Breit–Wigner fit of a binned mass spectrum
1. Import the histogram as a `RooDataHist`. 2. Fit a Breit–Wigner. 3. Add a Gaussian resolution and refit.
4. Compare the mass with the PDG value (≈ 5.367 GeV for the $B^0_s$).

In [ ]:
f = ROOT.TFile.Open("../../examples/lhcb-b0-ppbar/data/b0_invariant_mass.root")
h = f.Get("massaB0")
h.SetDirectory(0)
f.Close()

mass = ROOT.RooRealVar("mass", "mass [GeV]", 5.0, 6.0)
dh   = ROOT.RooDataHist("dh", "dh", ROOT.RooArgList(mass), h)

mean_bw  = ROOT.RooRealVar("mean_bw", "mass", 5.36, 5.2, 5.5)
width_bw = ROOT.RooRealVar("width_bw", "width", 0.1, 0.001, 0.5)
bw       = ROOT.RooBreitWigner("bw", "bw", mass, mean_bw, width_bw)
res_bw   = bw.fitTo(dh, Save=True, PrintLevel=-1)

frame_b = mass.frame(Title="Breit-Wigner fit")
dh.plotOn(frame_b)
bw.plotOn(frame_b)
c_b = ROOT.TCanvas("c_b", "c_b", 800, 500)
frame_b.Draw()
c_b.Draw()
mean_bw.Print(); width_bw.Print()

In [ ]:
check("BW fit converged", res_bw is not None and res_bw.status() == 0)
check("mass within 30 MeV of PDG (5.367 GeV)", abs(mean_bw.getVal() - 5.367) < 0.03)

### Add a Gaussian resolution
`RooVoigtian` is the analytic convolution Breit–Wigner ⊗ Gaussian.
Fix the resolution at σ_res = 30 MeV (it cannot be floated together with the width: why not?) and refit.

In [ ]:
mean_v  = ROOT.RooRealVar("mean_v", "mass", 5.36, 5.2, 5.5)
width_v = ROOT.RooRealVar("width_v", "width", 0.1, 0.001, 0.5)
sig_res = ROOT.RooRealVar("sig_res", "resolution", 0.03)   # fixed value, no range
voigt   = ROOT.RooVoigtian("voigt", "voigt", mass, mean_v, width_v, sig_res)
res_v   = voigt.fitTo(dh, Save=True, PrintLevel=-1)

frame_v = mass.frame(Title="Voigtian = BW (x) Gaussian")
dh.plotOn(frame_v)
voigt.plotOn(frame_v)
c_v = ROOT.TCanvas("c_v", "c_v", 800, 500)
frame_v.Draw()
c_v.Draw()
print("BW width:", width_bw.getVal(), " Voigtian BW width:", width_v.getVal())

In [ ]:
check("Voigtian fit converged", res_v is not None and res_v.status() == 0)

**Questions B1.** (a) How far is your mass from the PDG value, in units of the fit error?
(b) The natural width of the $B^0_s$ is of order 10⁻¹² GeV. Is the fitted width physical? What does it measure?
(c) Why do width and resolution cannot be fitted together here?

## B2 · The central limit theorem, and its tails
Sum of N independent uniform variables in [0,1]. The sum has mean N/2 and variance N/12.
1. Fit a Gaussian with RooFit to 10 000 sums (N = 10).
2. Count the fraction of 2 000 000 sums outside ±kσ (k = 1…4) and compare with the Gaussian expectation, for N = 2, 5, 10, 20.

In [ ]:
import math
rng = np.random.default_rng(MATRICOLA)
NEVT = 2_000_000

def make_sums(N):
    s = np.zeros(NEVT)
    for _ in range(N):
        s += rng.random(NEVT)
    return s

def fit_gauss(sums, N, nfit=10000):
    """Fit a RooGaussian to the first nfit sums. Return (mean_var, sigma_var, result)."""
    S  = ROOT.RooRealVar("S", "sum", 0, N)
    ds = ROOT.RooDataSet("ds", "ds", ROOT.RooArgSet(S))
    for v in sums[:nfit]:
        S.setVal(v)
        ds.add(ROOT.RooArgSet(S))
    m  = ROOT.RooRealVar("m", "m", N / 2, 0, N)
    sg = ROOT.RooRealVar("sg", "sg", math.sqrt(N / 12), 0.01, N)
    g  = ROOT.RooGaussian("g", "g", S, m, sg)
    res = g.fitTo(ds, Save=True, PrintLevel=-1)
    return m, sg, res

In [ ]:
sums10 = make_sums(10)
out = fit_gauss(sums10, 10)
check("fit_gauss implemented", out is not None)
m, sg, res = out
print(f"N=10: mean = {m.getVal():.3f} +- {m.getError():.3f}  (expect 5),  sigma = {sg.getVal():.3f} +- {sg.getError():.3f}  (expect {math.sqrt(10/12):.3f})")
check("mean compatible with N/2", abs(m.getVal() - 5) < 4 * m.getError())
check("sigma compatible with sqrt(N/12)", abs(sg.getVal() - math.sqrt(10 / 12)) < 4 * sg.getError())

In [ ]:
# Tail table: fraction outside +-k sigma, measured vs Gaussian
print(" N   k    measured      gauss      ratio")
for N in (2, 5, 10, 20):
    s = make_sums(N)
    z = (s - N / 2) / math.sqrt(N / 12)
    for k in (1, 2, 3, 4):
        measured = np.mean(np.abs(z) > k)
        expected = math.erfc(k / math.sqrt(2))
        print(f"{N:3d}  {k}   {measured:.3e}   {expected:.3e}   {measured/expected:6.2f}")

**Questions B2.** (a) For which (N, k) does the Gaussian tail probability fail by more than 20 %?
(b) For N = 2 the sum is triangular and has a hard edge: what happens beyond ±2.45σ?
(c) What does this teach about quoting a "5σ" result from a Gaussian approximation?